Notebook 4 —Écriture Parquet (et chargement PostgreSQL bonus)

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TradeCorp_Nettoyage") \
    .getOrCreate()

df_customers = spark.read.parquet("../data/tmp/customers")
df_orders= spark.read.parquet("../data/tmp/orders")
df_order_details= spark.read.parquet("../data/tmp/order_details")
df_products= spark.read.parquet("../data/tmp/products")
df_employees= spark.read.parquet("../data/tmp/employees")
df_categories = spark.read.csv("/home/jovyan/data/categories.csv", header=True, inferSchema=True)
df_suppliers = spark.read.csv("/home/jovyan/data/suppliers.csv", header=True, inferSchema=True)
df_shippers = spark.read.csv("/home/jovyan/data/shippers.csv", header=True, inferSchema=True)
df_orders_enriched = spark.read.parquet("/home/jovyan/data/output/orders_enriched.parquet")

print("---Spark est activé et toutes les tables sont chargées---")

---Spark est activé et toutes les tables sont chargées---


Q33 — Relire le Parquet

In [2]:
# lecture du fichier 
df_orders_enriched_parquet = spark.read.parquet("/home/jovyan/data/output/orders_enriched.parquet")

# vérification du nombre de lignes
print(f"Le nombre de lignes dans le fichier df_orders_enriched est de : {df_orders_enriched.count()}")
print(f"Le nombre de lignes dans le fichier df_orders_enriched_parquet est de : {df_orders_enriched_parquet.count()}")

# comparaison entre les 2 fichiers 
if df_orders_enriched_parquet.count() == df_orders_enriched.count():
    print("Le nombre de lignes est parfaitement identique entre l'original et le Parquet.")
else:
    print("Le nombre de lignes est différent.")
    
# vérification des schémas
if df_orders_enriched_parquet.dtypes == df_orders_enriched.dtypes:
    print("Les schémas sont parfaitement conservés.")
else:
    print("Les types de données ont été modifiés.")

Le nombre de lignes dans le fichier df_orders_enriched est de : 893
Le nombre de lignes dans le fichier df_orders_enriched_parquet est de : 893
Le nombre de lignes est parfaitement identique entre l'original et le Parquet.
Les schémas sont parfaitement conservés.


Q34 — Comparer CSV vs Parquet

Les fichiers CSV sont beaucoup plus lourds que les parquets.
Le CSV est un format simple et lisible, mais il est lourd et lent pour de gros volumes de données.
Le Parquet, lui, est optimisé, compressé et beaucoup plus rapide, ce qui le rend idéal pour le Big Data et les analyses.

Q35 — Partitionnement

In [3]:
# écriture du DF partitionné
(df_orders_enriched.write
     .mode("overwrite")
     .partitionBy("customer_country")
     .parquet("/home/jovyan/data/output/orders_partitioned.parquet")
)

print("Le dataframe est partitionné!")


Le dataframe est partitionné!


In [4]:
# Affichage de la strucuture des dossiers 
!ls -R /home/jovyan/data/output/orders_partitioned.parquet

/home/jovyan/data/output/orders_partitioned.parquet:
 _SUCCESS		      'customer_country=ITALY'
'customer_country=ARGENTINA'  'customer_country=MEXICO'
'customer_country=AUSTRIA'    'customer_country=NORWAY'
'customer_country=BELGIUM'    'customer_country=POLAND'
'customer_country=BRAZIL'     'customer_country=PORTUGAL'
'customer_country=CANADA'     'customer_country=SPAIN'
'customer_country=DENMARK'    'customer_country=SWEDEN'
'customer_country=FINLAND'    'customer_country=SWITZERLAND'
'customer_country=FRANCE'     'customer_country=UK'
'customer_country=GERMANY'    'customer_country=USA'
'customer_country=IRELAND'    'customer_country=VENEZUELA'

'/home/jovyan/data/output/orders_partitioned.parquet/customer_country=ARGENTINA':
part-00000-1d419cb1-acaf-4cbf-868c-b711333bbc74.c000.snappy.parquet

'/home/jovyan/data/output/orders_partitioned.parquet/customer_country=AUSTRIA':
part-00000-1d419cb1-acaf-4cbf-868c-b711333bbc74.c000.snappy.parquet

'/home/jovyan/data/output/orders_partition

Q36 — Chargement PostgreSQL via JDBC

In [5]:
# définition des paramètres 
jdbc_url = "jdbc:postgresql://tradecorp_postgres:5432/tradecorp"
properties = {
    "user": "tradecorp",
    "password": "tradecorp",
    "driver": "org.postgresql.Driver"
}

# écriture df_orders_enriched dans PostgreSQL via JDBC dans une table orders_enriched
(df_orders_enriched.write
    .mode("overwrite")
    .jdbc(url=jdbc_url, table="orders_enriched", properties=properties)
)

print("Données envoyées dans PostgreSQL !")

Données envoyées dans PostgreSQL !
